# tomato-agent: stages 1-3b on a Colab GPU
Runtime > Change runtime type > **T4 GPU** before you start.
Upload `tomato-agent.zip` (code only: leave out `.venv`, `smoke`, `runs`, `PlantVillage-Dataset`, `PlantDoc-Dataset`) to Google Drive. **No images are uploaded anywhere**: Colab downloads the datasets itself from the public GitHub repos in section 2, and results stay in your Drive.
Download `tomato_results.zip` at the end, unzip it into the repo on your laptop, then run stages 4-8 there.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put tomato-agent.zip in My Drive, or change this path
ZIP = '/content/drive/MyDrive/tomato-agent.zip'
!rm -rf /content/tomato-agent && mkdir -p /content/tomato-agent
!unzip -q -o "$ZIP" -d /content/tomato-agent
import os, glob
# if the zip contains a single top-level folder, step into it
root = '/content/tomato-agent'
sub = [d for d in glob.glob(root+'/*') if os.path.isdir(d)]
if not os.path.exists(root+'/run_all.sh') and len(sub)==1: root = sub[0]
os.chdir(root); print(os.getcwd()); print(os.listdir('.'))

In [ ]:
!pip install -q timm imagehash onnx onnxruntime scipy pandas matplotlib pytest
!python -m pytest -q

## 2. Data (tomato only, ~1.1 GB)

In [ ]:
!bash scripts/download_data.sh

## 3. Stages 1-2 (manifests, dedup, quality thresholds)
Compare with `docs/reference_run/`.

In [ ]:
PV='PlantVillage-Dataset'; PD='PlantDoc-Dataset'; M='manifests'; A='artifacts'; R='runs'
DATA=f"--pv-root {PV} --pd-root {PD} --pv-manifest {M}/pv_manifest_dedup.csv --pd-manifest {M}/pd_manifest.csv"
!python scripts/prepare_data.py manifests --pv-root $PV --pd-root $PD --out $M
!python scripts/prepare_data.py dedup --pv-root $PV --manifest $M/pv_manifest.csv --out $M
!python scripts/set_quality_thresholds.py {DATA} --out $A --examples 3

## 4. Stage 3: train
Pilot first (one backbone, one seed). Training resumes: finished runs have `metrics.json` and are skipped, so you can re-run this cell after a disconnect.
For the full study use `BACKBONES = 'efficientnet_b0 resnet50 vit_small_patch16_224'` and `SEEDS = '0 1 2 3 4'` (may need several Colab sessions; copy `runs/` to Drive between them).

In [ ]:
BACKBONES = 'efficientnet_b0'
SEEDS = '0'
# keep runs on Drive so a disconnect does not lose them
!mkdir -p /content/drive/MyDrive/tomato_runs
!rm -rf runs && ln -s /content/drive/MyDrive/tomato_runs runs
!python scripts/train_diagnosis.py {DATA} --backbones {BACKBONES} --seeds {SEEDS} --out $R --amp --workers 2

In [ ]:
!python scripts/predict_conditions.py {DATA} --runs $R --quality $A/quality_thresholds.json --workers 2

## 5. Package results for the laptop

In [ ]:
!zip -qr /content/tomato_results.zip runs manifests artifacts dataset_versions.txt
from google.colab import files
files.download('/content/tomato_results.zip')